In [1]:
%%capture
!pip install bertopic
import re
import pandas as pd
from datetime import datetime
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from bertopic import BERTopic
!pip install googletrans==3.1.0a0
# !pip install deep-translator==1.11.4
# !pip install lingua-language-detector==2.0.2
# !pip install python-iso639==2024.1.2
# from deep_translator import GoogleTranslator, LibreTranslator, MyMemoryTranslator
# from lingua import Language, LanguageDetectorBuilder
# import iso639
# DETECTOR = LanguageDetectorBuilder.from_all_languages().with_preloaded_language_models().build()
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

2024-04-18 16:16:53.830573: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-18 16:16:53.830732: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-18 16:16:53.974384: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
from googletrans import Translator
translator = Translator(service_urls=[
    'translate.googleapis.com',
  'translate.google.com',
  'translate.google.co.kr',
  'translate.google.co.in',
  'translate.google.co.uk',
])

# language tranlator utility
def trans_msg(msg):
#     if type(msg) != str:
#         return
    trans_m = translator.translate(str(msg), dest='en')
    return str(trans_m.text)

In [3]:
%%capture
# !pip install -U pip setuptools wheel
# !pip install -U spacy
!python -m spacy download ru_core_news_sm
!python -m spacy download zh_core_web_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download de_core_news_sm
!python -m spacy download it_core_news_sm
!python -m spacy download xx_ent_wiki_sm
!python -m spacy download pl_core_news_sm
!python -m spacy download pt_core_news_sm
!python -m spacy download ro_core_news_sm
!python -m spacy download es_core_news_sm
!python -m spacy download uk_core_news_sm
!python -m spacy download sl_core_news_sm
!python -m spacy download sv_core_news_sm

import spacy
import gensim
from gensim.parsing.preprocessing import remove_stopwords, STOPWORDS

stop_words = stopwords.words('russian')
stop_words.extend(['nan','что', 'это', 'так', 'вот', 'быть', 'как', 'в', '—', 'к', 'на'])
for i in stopwords.fileids():
    stop_words.extend(stopwords.words(str(i)))

en = spacy.load('en_core_web_sm')
sw_spacy = en.Defaults.stop_words
stop_words.extend(list(sw_spacy))

ru = spacy.load('ru_core_news_sm')
sw_spacy_ru = ru.Defaults.stop_words
stop_words.extend(list(sw_spacy_ru))

zh = spacy.load('zh_core_web_sm')
sw_spacy_zh = zh.Defaults.stop_words
stop_words.extend(list(sw_spacy_zh))

fr = spacy.load('fr_core_news_sm')
sw_spacy_fr = fr.Defaults.stop_words
stop_words.extend(list(sw_spacy_fr))

de = spacy.load('de_core_news_sm')
sw_spacy_de = de.Defaults.stop_words
stop_words.extend(list(sw_spacy_de))

it = spacy.load('it_core_news_sm')
sw_spacy_it = it.Defaults.stop_words
stop_words.extend(list(sw_spacy_it))

xx = spacy.load('xx_ent_wiki_sm')
sw_spacy_xx = xx.Defaults.stop_words
stop_words.extend(list(sw_spacy_xx))

pl = spacy.load('pl_core_news_sm')
sw_spacy_pl = pl.Defaults.stop_words
stop_words.extend(list(sw_spacy_pl))

pt = spacy.load('pt_core_news_sm')
sw_spacy_pt = pt.Defaults.stop_words
stop_words.extend(list(sw_spacy_pt))

ro = spacy.load('ro_core_news_sm')
sw_spacy_ro = ro.Defaults.stop_words
stop_words.extend(list(sw_spacy_ro))

es = spacy.load('es_core_news_sm')
sw_spacy_es = es.Defaults.stop_words
stop_words.extend(list(sw_spacy_es))

uk = spacy.load('uk_core_news_sm')
sw_spacy_uk = uk.Defaults.stop_words
stop_words.extend(list(sw_spacy_uk))

sl = spacy.load('sl_core_news_sm')
sw_spacy_sl = sl.Defaults.stop_words
stop_words.extend(list(sw_spacy_sl))

sv = spacy.load('sv_core_news_sm')
sw_spacy_sv = sv.Defaults.stop_words
stop_words.extend(list(sw_spacy_sv))

stop_words.extend(list(STOPWORDS))
stop_words.extend(['suscribirse', 'SuscrÃbete','telegra','subscribe','good','bad','better'])

# Yellow DF

In [4]:
mos = pd.read_csv('/kaggle/input/cluster-msgs-data/yellowdf.csv')
mos["just_date"] = pd.to_datetime(mos.date).dt.date

mos['msg_without_stopwords'] = mos['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mos

timestamps = mos.date.to_list()
texts1 = mos.msg_without_stopwords.to_list()

In [5]:
from bertopic import BERTopic

topic_model = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=25)
topics, _ = topic_model.fit_transform(texts1); len(topic_model.get_topic_info())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/389 [00:00<?, ?it/s]

39

In [6]:
freq = topic_model.get_topic_info()
freq['translated_Name'] = freq.Name.apply(trans_msg)
freq['translated_Representation'] = freq.Representation.apply(trans_msg)
freq['translated_Representative_Docs'] = freq.Representative_Docs.apply(trans_msg)

freq.head(10)

,Topic,Count,Name,Representation,Representative_Docs,translated_Name,translated_Representation,translated_Representative_Docs
0,-1,4632,-1_rusia_infodefense_infodefenseespañol_suscri...,"[rusia, infodefense, infodefenseespañol, suscr...",[Rusia prevenir guerra nuclear 🌎🌍🌏 luz crecien...,-1_russia_infodefense_infodefenseespañol_subsc...,"['russia', 'infodefense', 'infodefenseespañol'...",['Russia prevent nuclear war 🌎🌍🌏 growing light...
1,0,3379,0_explica_cosas__,"[explica, cosas, , , , , , , , ]","[, , ]",0_explains_things__,"['explains', 'things', '', '', '', '', '', '',...","['', '', '']"
2,1,687,1_ucrania_ucranianos_ucraniano_kiev,"[ucrania, ucranianos, ucraniano, kiev, ucrania...","[🇺🇦 ""Ucrania planea Guerra Mundial"": Kiev plan...",1_Ukraine_Ukrainians_Ukrainian_Kiev,"['Ukraine', 'Ukrainians', 'Ukrainian', 'kyiv',...","['🇺🇦 ""Ukraine plans World War"": kyiv plans to ..."
3,2,380,2_israel_gaza_israelí_franja,"[israel, gaza, israelí, franja, hamás, palesti...",[❗️🇵🇸🇮🇱 bombardeo Franja Gaza Fuerza Aérea Art...,2_israel_gaza_israeli_strip,"['israel', 'gaza', 'Israeli', 'strip', 'hamas'...",['❗️🇵🇸🇮🇱 bombardment Gaza Strip Air Force Arti...
4,3,356,3_putin_presidente_vladímir_vladimir,"[putin, presidente, vladímir, vladimir, rusia,...","[❗️En ataque Kremlin, punto culminante, cereza...",3_putin_president_vladimir_vladimir,"['putin', 'president', 'vladímir', 'vladimir',...","['❗️In Kremlin attack, climax, Anglo-Saxon che..."
5,4,297,4_suscribirse_infodefenseespañol_infodefense_para,"[suscribirse, infodefenseespañol, infodefense,...",[¡Buenas noches! ✔️Para suscribirse: 📱 InfoDef...,4_subscribe_infodefenseespañol_infodefense_para,"['subscribe', 'infodefensa español', 'infodefe...",['Good night! ✔️To subscribe: 📱 InfoDefenseESP...
6,5,257,5_nazis_nazi_nazismo_ver,"[nazis, nazi, nazismo, ver, hitler, guerra, fa...",[🇺🇦En Ucrania renombran calle honor colaborado...,5_nazis_nazi_nazismo_ver,"['nazis', 'nazi', 'nazism', 'see', 'hitler', '...",['🇺🇦In Ukraine street is renamed honor Nazi co...
7,6,171,6_rusia_occidente_rusos_rusa,"[rusia, occidente, rusos, rusa, ruso, occident...",[🇷🇺Las declaraciones occidentales Rusia supues...,6_russia_west_russians_russian,"['russia', 'west', 'russians', 'russian', 'rus...",['🇷🇺Western statements Russia supposedly plans...
8,7,162,7_terrorista_atentado_terroristas_terrorismo,"[terrorista, atentado, terroristas, terrorismo...",[🇷🇺 mundo solidariza Rusia atentado terrorista...,7_terrorist_attack_terrorists_terrorism,"['terrorist', 'attack', 'terrorists', 'terrori...",['🇷🇺 world shows solidarity Russia terrorist a...
9,8,145,8_alemania_alemán_scholz_alemanes,"[alemania, alemán, scholz, alemanes, ucrania, ...","[📌🇩🇪En Alemania, personas admitieron haberse v...",8_germany_german_scholz_germans,"['germany', 'german', 'scholz', 'germans', 'uk...","['📌🇩🇪In Germany, people admitted to becoming d..."


In [7]:
topics_over_time = topic_model.topics_over_time(docs=texts1,
                                                timestamps=timestamps,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=(len(freq)+1))

20it [00:08,  2.43it/s]


In [8]:
freq.to_csv('yellowdf_topics_BERTopic.csv', index=False)

# Blue DF

In [9]:
mosB = pd.read_csv('/kaggle/input/cluster-msgs-data/bluedf.csv')
mosB["just_date"] = pd.to_datetime(mosB.date).dt.date

mosB['msg_without_stopwords'] = mosB['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mosB

timestampsB = mosB.date.to_list()
textsB = mosB.msg_without_stopwords.to_list()

In [10]:
from bertopic import BERTopic

topic_modelB = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=50)
topicsB, _ = topic_modelB.fit_transform(textsB); len(topic_modelB.get_topic_info())

Batches:   0%|          | 0/1183 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

38

In [11]:
freqB = topic_modelB.get_topic_info()
freqB['translated_Name'] = freqB.Name.apply(trans_msg)
freqB['translated_Representation'] = freqB.Representation.apply(trans_msg)
freqB['translated_Representative_Docs'] = freqB.Representative_Docs.apply(trans_msg)

freqB.head(10)

,Topic,Count,Name,Representation,Representative_Docs,translated_Name,translated_Representation,translated_Representative_Docs
0,-1,6380,-1_000_шт_сбор_руб,"[000, шт, сбор, руб, майора, 10, штук, dji, сб...",[#Сбор пять подразделений 205-й отдельной мото...,-1_000_pcs_collection_rub,"['000', 'pieces', 'collection', 'rub', 'major'...",['#Сбор пять подразделений 205-й отдельной мот...
1,0,9758,0_майора_область_всу_россии,"[майора, область, всу, россии, области, пво, с...","[Белгородская область, губернатор. результате ...",0_major_region_all_russia,"['major', 'region', 'AAF', 'Russia', 'region',...","['Belgorod region, governor. A civilian was in..."
2,1,8743,1_vae_victis_shaman_,"[vae, victis, shaman, , , , , , , ]","[, , ]",1_woe_of_the_defeated_shaman_,"['woe', 'victims', 'shaman', '', '', '', '', '...","['', '', '']"
3,2,4939,2_майора_россии_то_всу,"[майора, россии, то, всу, украины, противника,...",[новой цели ЦИПСО фоне наступательных действий...,2_major_of_russia_to_vsu,"['major', 'russia', 'to', 'vsu', 'ukraine', 'e...",['новой цели ЦИПСО фоне наступательных действи...
4,3,2633,3_нет_всу_россии_разрушений,"[нет, всу, россии, разрушений, противника, пос...","[Губернатор Белгородской области, августа: ⠀ 📍...",3_no_destruction_in_all_Russia,"['no', 'APU', 'Russia', 'destruction', 'enemy'...","['Губернатор Белгородской области, августа: ⠀ ..."
5,4,804,4_всу_направлении_россии_вс,"[всу, направлении, россии, вс, на, бои, против...",[#Сводка утро 19 января 2024 г. ▪️В Крынках Хе...,4_all_direction_russia_sun,"['vsu', 'direction', 'russia', 'sun', 'on', 'b...",['#Сводка утро 19 января 2024 г. ▪️В Крынках Х...
6,5,798,5_отбой_ночи_доброй_подъём,"[отбой, ночи, доброй, подъём, построение, добр...","[Доброй ночи, товарищи! ОТБОЙ! майора вами!, л...",5 lights out good morning,"['lights out', 'night', 'good', 'rise', 'forma...","['Good night, comrades! CALL OUT! Major by you..."
7,6,478,6_romanov_92_2023_россия_romanov,"[romanov_92, 2023, россия, romanov, область, л...","[22.06.2023 Пологовский р-н, Запорожская облас...",6_romanov_92_2023_russia_romanov,"['romanov_92', '2023', 'russia', 'romanov', 'r...","['06/22/2023 Pologovsky district, Zaporozhye r..."
8,7,314,7_leopard_сша_000_майора,"[leopard, сша, 000, майора, patriot, https, ст...","[⚡️⚡️⚡️⚡️⚡️ ПТУРистов, осуществляющих боевую р...",7_leopard_usa_000_major,"['leopard', 'usa', '000', 'major', 'patriot', ...",['⚡️⚡️⚡️⚡️⚡️ ATGM players carrying out combat ...
9,8,247,8_100_200_тыс_человек,"[100, 200, тыс, человек, рублей, всу, 50, сбор...",[Пятничная математика Двух майоров Сбор 291 по...,8_100_200_thousand_people,"['100', '200', 'thousand', 'person', 'rubles',...",['Friday mathematics of Two Majors Collection ...


In [12]:
topics_over_timeB = topic_modelB.topics_over_time(docs=textsB,
                                                timestamps=timestampsB,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)

topic_modelB.visualize_topics_over_time(topics_over_timeB, top_n_topics=(len(freqB)+1))

20it [00:31,  1.56s/it]


In [13]:
freqB.to_csv('bluedf_topics_BERTopic.csv', index=False)

# Green DF

In [14]:
mosG = pd.read_csv('/kaggle/input/cluster-msgs-data/greendf.csv')
mosG["just_date"] = pd.to_datetime(mosG.date).dt.date

mosG['msg_without_stopwords'] = mosG['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mosG

timestampsG = mosG.date.to_list()
textsG = mosG.msg_without_stopwords.to_list()

In [15]:
from bertopic import BERTopic

topic_modelG = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=250)
topicsG, _ = topic_modelG.fit_transform(textsG); len(topic_modelG.get_topic_info())

Batches:   0%|          | 0/7337 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

74

In [16]:
freqG = topic_modelG.get_topic_info()
freqG['translated_Name'] = freqG.Name.apply(trans_msg)
freqG['translated_Representation'] = freqG.Representation.apply(trans_msg)
freqG['translated_Representative_Docs'] = freqG.Representative_Docs.apply(trans_msg)

freqG.head(10)

,Topic,Count,Name,Representation,Representative_Docs,translated_Name,translated_Representation,translated_Representative_Docs
0,-1,45681,-1_всу_рф_украины_украина,"[всу, рф, украины, украина, россии, украине, г...",[⚡️ Сводка Министерства обороны Российской Фед...,-1_vsu_rf_ukraine_ukraine,"['vsu', 'rf', 'ukraine', 'ukraine', 'russia', ...",['⚡️ Сводка Министерства обороны Российской Фе...
1,0,39525,0_comments_clown_weide_robert,"[comments, clown, weide, robert, directed, sav...","[, , ]",0_comments_clown_weide_robert,"['comments', 'clown', 'weide', 'robert', 'dire...","['', '', '']"
2,1,25600,1_украины_всу_украина_россии,"[украины, всу, украина, россии, украине, облас...",[Заявления Путина пресс-подходе итогам года: ❗...,1_ukraine_vsu_ukraine_russia,"['ukraine', 'vsu', 'ukraine', 'russia', 'ukrai...",['Заявления Путина пресс-подходе итогам года: ...
3,2,24070,2_украины_то_россии_фейк,"[украины, то, россии, фейк, всу, украине, том,...","[Спецоперация, 14 июля. Главное РИА Новости: ▪...",2_Ukraine_to_Russia_fake,"['ukraine', 'to', 'russia', 'fake', 'vsu', 'uk...","['Спецоперация, 14 июля. Главное РИА Новости: ..."
4,3,18324,3_области_сообщают_путин_рф,"[области, сообщают, путин, рф, взрывах, заявил...",[⚡Сообщают взрывах Николаеве. области объявлен...,3 regions report Putin of the Russian Federation,"['regions', 'report', 'Putin', 'rf', 'explosio...",['⚡They report explosions in Nikolaev. region ...
5,4,17358,4_сообщают_области_заявил_сообщает,"[сообщают, области, заявил, сообщает, сообщил,...",[🚀 Украинские ТГ-каналы сообщают пусках ракет ...,4 report areas stated reports,"['report', 'regions', 'stated', 'reports', 're...",['🚀 Ukrainian TG channels report missile launc...
6,5,10771,5_видео_всу_то_украине,"[видео, всу, то, украине, украины, украина, ро...","[Голос Мордора приятное, честное прямое мнение...",5_video_all_to_ukraine,"['video', 'vsu', 'to', 'ukraine', 'ukraine', '...","['The Voice of Mordor is a pleasant, honest, d..."
7,6,5765,6_то_украины_год_что,"[то, украины, год, что, украина, нет, украине,...","[❗️Владимир Зеленский заявил, Украине времени ...",6_that year of Ukraine,"['to', 'Ukraine', 'year', 'what', 'Ukraine', '...",['❗️Vladimir Zelensky said Ukraine is in time ...
8,7,2480,7_опрошенных_украинцев_года_опроса,"[опрошенных, украинцев, года, опроса, считают,...",[📉 Общество поддерживает действия президента д...,7 Ukrainians interviewed in the survey year,"['surveyed', 'Ukrainians', 'year', 'poll', 'co...",['📉 The society supports the president's actio...
9,8,2445,8_год_ссср_года_году,"[год, ссср, года, году, войска, родился, 1941,...",[🕘 января Новый год наступил! 1810 год Императ...,8_year_ussr_year_year,"['year', 'USSR', 'year', 'year', 'troops', 'bo...",['🕘 January New Year has arrived! 1810 Emperor...


In [17]:
topics_over_timeG = topic_modelG.topics_over_time(docs=textsG,
                                                timestamps=timestampsG,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)

topic_modelG.visualize_topics_over_time(topics_over_timeG, top_n_topics=(len(freqG)+1))

20it [02:33,  7.69s/it]


In [18]:
freqG.to_csv('greendf_topics_BERTopic.csv', index=False)

# Pink DF

In [19]:
mosP = pd.read_csv('/kaggle/input/cluster-msgs-data/pinkdf.csv')
mosP["just_date"] = pd.to_datetime(mosP.date).dt.date

mosP['msg_without_stopwords'] = mosP['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mosP

timestampsP = mosP.date.to_list()
textsP= mosP.msg_without_stopwords.to_list()

In [20]:
from bertopic import BERTopic

topic_modelP = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=260)
topicsP, _ = topic_modelP.fit_transform(textsP); len(topic_modelP.get_topic_info())

Batches:   0%|          | 0/7940 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

66

In [21]:
freqP = topic_modelP.get_topic_info()
freqP['translated_Name'] = freqP.Name.apply(trans_msg)
freqP['translated_Representation'] = freqP.Representation.apply(trans_msg)
freqP['translated_Representative_Docs'] = freqP.Representative_Docs.apply(trans_msg)

freqP.head(10)

,Topic,Count,Name,Representation,Representative_Docs,translated_Name,translated_Representation,translated_Representative_Docs
0,-1,39903,-1_область_республика_live_https,"[область, республика, live, https, смотрите, р...",[⚡️Коронавирус России выявлено 25 487 новых сл...,-1_region_of_the_republic_live_https,"['region', 'republic', 'live', 'https', 'look'...",['⚡️Коронавирус России выявлено 25 487 новых с...
1,0,57364,0_россии_рф_всу_сша,"[россии, рф, всу, сша, области, года, украины,...",[‼️‼️‼️Главное брифинга Минобороны: 📌На Донецк...,0_russia_rf_vsu_usa,"['russia', 'rf', 'vsu', 'usa', 'region', 'goda...",['‼️‼️‼️Главное брифинга Минобороны: 📌На Донец...
2,1,32732,1_абхаз_ахра_авидзба_пятнашка,"[абхаз, ахра, авидзба, пятнашка, позывной, liv...","[, , ]",1_abkhaz_akhra_avidzba_pytnashka,"['abkhaz', 'akhra', 'avidzba', 'tag', 'call si...","['', '', '']"
3,2,30469,2_то_россии_сша_время,"[то, россии, сша, время, том, сегодня, людей, ...",[Западную элиту шокирует военное наступление Р...,2_Russia_USA_time,"['to', 'russia', 'usa', 'time', 'tom', 'today'...",['Западную элиту шокирует военное наступление ...
4,3,16701,3_путин_видео_сегодня_сша,"[путин, видео, сегодня, сша, то, россии, песко...",[ПЕСКОВ Песков личной встрече Путина Зеленског...,3 Putin video today USA,"['Putin', 'video', 'today', 'USA', 'before', '...",['PESKOV Peskov's personal meeting with Putin ...
5,4,16027,4_путин_песков_риа_россии,"[путин, песков, риа, россии, области, сша, нов...",[⚡️⚡️⚡️⚡️ Воздушная тревога объявлена Киеве Ки...,4_putin_peskov_ria_russia,"['putin', 'peskov', 'ria', 'russia', 'region',...",['⚡️⚡️⚡️⚡️ An air alert has been declared in K...
6,5,7045,5_путин_риа_новости_песков,"[путин, риа, новости, песков, россии, сша, зая...","[Песков подтвердил, Путин Эрдоган поговорили т...",5 putin ria news sands,"['putin', 'ria', 'news', 'peskov', 'russia', '...",['Peskov confirmed that Putin and Erdogan spok...
7,6,3399,6_live_соловьёв_эфире_эфир,"[live, соловьёв, эфире, эфир, смотрите, соловь...",[💡ЭФИР ЛАБИРИНТ КАРНАУХОВА СОЛОВЬЁВ LIVE Серге...,6 live nightingales broadcast,"['live', 'soloviev', 'ether', 'ether', 'look',...",['💡BREAKING THE LABYRINTH OF KARNAUKHOV SOLOVI...
8,7,2502,7_telegram_подписывайся_соловьёв_россии,"[telegram, подписывайся, соловьёв, россии, сол...","[📹Ну почти. Подписывайся Telegram СОЛОВЬЁВ!, ⚡...",7 telegram subscribe nightingales russia,"['telegram', 'subscribe', 'soloviev', 'russia'...","['📹Well, almost. Subscribe Telegram SOLOVYOV!'..."
9,8,2467,8_me_https_solovievlive_dimsmirnov175,"[me, https, solovievlive, dimsmirnov175, vityz...","[Сильно https://t.me/SolovievLive/73291, Спаси...",8_me_https_solovievlive_dimsmirnov175,"['me', 'https', 'solovievlive', 'dimsmirnov175...","['Strongly https://t.me/SolovievLive/73291', '..."


In [22]:
topics_over_timeP = topic_modelP.topics_over_time(docs=textsP,
                                                timestamps=timestampsP,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)

topic_modelP.visualize_topics_over_time(topics_over_timeP, top_n_topics=(len(freqP)+1))

20it [03:12,  9.61s/it]


In [23]:
freqP.to_csv('pinkdf_topics_BERTopic.csv', index=False)

# Red DF

In [24]:
mosR = pd.read_csv('/kaggle/input/cluster-msgs-data/reddf.csv')
mosR["just_date"] = pd.to_datetime(mosR.date).dt.date

mosR['msg_without_stopwords'] = mosR['cleaned_message'].apply(lambda x: ' '.join([word for word in str(x).split() if word.lower() not in (stop_words)]))
# mosR

timestampsR = mosR.date.to_list()
textsR = mosR.msg_without_stopwords.to_list()

/tmp/ipykernel_24/2057609697.py:1: DtypeWarning:

Columns (12,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.



In [25]:
from bertopic import BERTopic

topic_modelR = BERTopic(verbose=True, embedding_model="paraphrase-MiniLM-L12-v2", min_topic_size=300)
topicsR, _ = topic_modelR.fit_transform(textsR); len(topic_modelR.get_topic_info())

Batches:   0%|          | 0/11988 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

87

In [26]:
freqR = topic_modelR.get_topic_info(); freqR.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,64364,-1_https_co_россии_подписаться,"[https, co, россии, подписаться, ru, рф, iz, с...",[⚡️За последние сутки России выявлено 980 новы...
1,0,42882,0_путин_рф_коронавируса_песков,"[путин, рф, коронавируса, песков, коронавирусо...",[❗️🦠Все регионы Сибири ввели режим самоизоляци...
2,1,38300,1_рф_zvezdanews_всу_россии,"[рф, zvezdanews, всу, россии, направлении, сша...",[📋📋📋📋Поражение складов ракетно-артиллерийского...
3,2,35500,2_classico_fair_free_16,"[classico, fair, free, 16, , , , , , ]","[, , ]"
4,3,29772,3_iz_ru_подписаться_рф,"[iz, ru, подписаться, рф, владимир, сообщил, с...",[Песков: конкурентов Путина России 😎 Подписать...
5,4,26000,4_izvestia_владимир_заявил_россии,"[izvestia, владимир, заявил, россии, путин, рф...","[⚡️Шольц заявил, Евросоюз сегодня примет новые..."
6,5,24273,5_то_ru_iz_подписаться,"[то, ru, iz, подписаться, россии, рассказал, в...","[Жизнь Донецке, городах ДНР, напоминает лотере..."
7,6,10675,6_рф_человек_россии_коронавируса,"[рф, человек, россии, коронавируса, путин, мос...",[🔥Жилой дом Омской области полностью уничтожил...
8,7,10640,7_видео_то_сегодня_россии,"[видео, то, сегодня, россии, фото, лет, челове...",[Наблюдатели Камбоджи приехать выборы президен...
9,8,6401,8_co_https_клинтон_москве,"[co, https, клинтон, москве, сша, сирии, росси...",[Трамп заявил возможности сотрудничества США Р...


In [27]:
freqR = topic_modelR.get_topic_info()
freqR['translated_Name'] = freqR.Name.apply(trans_msg)
freqR['translated_Representation'] = freqR.Representation.apply(trans_msg)
freqR['translated_Representative_Docs'] = freqR.Representative_Docs.apply(trans_msg)

freqR.head(10)

,Topic,Count,Name,Representation,Representative_Docs,translated_Name,translated_Representation,translated_Representative_Docs
0,-1,64364,-1_https_co_россии_подписаться,"[https, co, россии, подписаться, ru, рф, iz, с...",[⚡️За последние сутки России выявлено 980 новы...,-1_https_co_russia_subscribe,"['https', 'co', 'russia', 'subscribe', 'ru', '...","['⚡️Over the past 24 hours, 980 new cases of c..."
1,0,42882,0_путин_рф_коронавируса_песков,"[путин, рф, коронавируса, песков, коронавирусо...",[❗️🦠Все регионы Сибири ввели режим самоизоляци...,0_putin_rf_coronavirus_sands,"['putin', 'rf', 'coronavirus', 'peskov', 'coro...",['❗️🦠All regions of Siberia have introduced a ...
2,1,38300,1_рф_zvezdanews_всу_россии,"[рф, zvezdanews, всу, россии, направлении, сша...",[📋📋📋📋Поражение складов ракетно-артиллерийского...,1_рф_zvezdanews_all_russia,"['rf', 'zvezdanews', 'vsu', 'russia', 'directi...",['📋📋📋📋Поражение складов ракетно-артиллерийског...
3,2,35500,2_classico_fair_free_16,"[classico, fair, free, 16, , , , , , ]","[, , ]",2_classico_fair_free_16,"['classico', 'fair', 'free', '16', '', '', '',...","['', '', '']"
4,3,29772,3_iz_ru_подписаться_рф,"[iz, ru, подписаться, рф, владимир, сообщил, с...",[Песков: конкурентов Путина России 😎 Подписать...,3_iz_ru_subscribe_рф,"['iz', 'ru', 'subscribe', 'rf', 'vladimir', 'r...",['Peskov: Putin's competitors to Russia 😎 Subs...
5,4,26000,4_izvestia_владимир_заявил_россии,"[izvestia, владимир, заявил, россии, путин, рф...","[⚡️Шольц заявил, Евросоюз сегодня примет новые...",4 izvestia Vladimir said to Russia,"['izvestia', 'Vladimir', 'stated', 'Russia', '...",['⚡️Scholz said that the European Union will t...
6,5,24273,5_то_ru_iz_подписаться,"[то, ru, iz, подписаться, россии, рассказал, в...","[Жизнь Донецке, городах ДНР, напоминает лотере...",5_to_ru_iz_subscribe,"['to', 'ru', 'iz', 'subscribe', 'Russia', 'tol...","['Жизнь Донецке, городах ДНР, напоминает лотер..."
7,6,10675,6_рф_человек_россии_коронавируса,"[рф, человек, россии, коронавируса, путин, мос...",[🔥Жилой дом Омской области полностью уничтожил...,6_rf_ people of Russia coronavirus,"['rf', 'person', 'russia', 'coronavirus', 'put...",['🔥A residential building in the Omsk region w...
8,7,10640,7_видео_то_сегодня_россии,"[видео, то, сегодня, россии, фото, лет, челове...",[Наблюдатели Камбоджи приехать выборы президен...,7_video_to_today_of_russia,"['video', 'then', 'today', 'Russia', 'photo', ...",['Cambodia observers to attend the Russian pre...
9,8,6401,8_co_https_клинтон_москве,"[co, https, клинтон, москве, сша, сирии, росси...",[Трамп заявил возможности сотрудничества США Р...,8_co_https_clinton_moscow,"['co', 'https', 'clinton', 'moscow', 'usa', 's...",['Trump announced the possibility of US cooper...


In [28]:
topics_over_timeR = topic_modelR.topics_over_time(docs=textsR,
                                                timestamps=timestampsR,
                                                global_tuning=True,
                                                evolution_tuning=True,
                                                nr_bins=20)

topic_modelR.visualize_topics_over_time(topics_over_timeR, top_n_topics=(len(freqR)+1))

20it [03:35, 10.77s/it]


In [29]:
freqR.to_csv('reddf_topics_BERTopic.csv', index=False)

# Scraping bridge channel's messages

# Similarity

### Misc

In [30]:
# topic_model.visualize_barchart(top_n_topics=51)

In [31]:
# topic_model.visualize_hierarchy(top_n_topics=5)

In [32]:
# freq.to_csv('yellowdf_topics_BERTopic.csv', index=False)